# CodeFlowLM - Prediction times measurements

This notebook tests CodeFlowLM inference time for a single prediction with both base learners (CodeT5+ and UniXCoder).

## Code preparation

In [ ]:
!git clone https://github.com/monilouise/codeflowlm_jitdl.git

In [ ]:
!git clone https://github.com/monilouise/PEFT4CC.git

In [ ]:
!git clone https://github.com/monilouise/codeflowlm.git

In [17]:
import sys
sys.path.append('codeflowlm')

In [ ]:
path = 'codeflowlm_jitdl/data'
full_features_train_file = f"{path}/ord_cross_features_train.pkl"
full_features_valid_file = f"{path}/ord_cross_features_valid.pkl"
full_features_test_file = f"{path}/ord_cross_features_test.pkl"
full_changes_train_file = f"{path}/ord_cross_changes_train_lst.pkl"
full_changed_valid_file = f"{path}/ord_cross_changes_valid_lst.pkl"
full_changes_test_file = f"{path}/ord_cross_changes_test_lst.pkl"
commit_guru_path = f"{path}/commit_guru/"
model_root = "." #path where the models will be saved
batch_classifier_dir = '.' #e.g: PEFT4CC

## Libraries installation

In [ ]:
!pip install river

In [ ]:
!pip install transformers==5.9.0

In [ ]:
!pip install huggingface_hub==1.16.4

In [ ]:
!pip install torchao==0.17.0

In [24]:
import codeflowlm.train
import codeflowlm.predict

## Training model for inference tests

### UniXCoder PreT

In [ ]:
!mkdir unixcoder-pret

In [ ]:
codeflowlm.train.train_project_with_lat_ver(batch_classifier_dir, path, 'unixcoder-pret/.', commit_guru_path,
                                            full_features_train_file, full_features_valid_file,
                                            full_features_test_file, full_changes_train_file,
                                            full_changed_valid_file, full_changes_test_file,
                                            project='commons-scxml', early_stop_metric='gmean',
                                            adjust_th=False, skewed_oversample=True,
                                            batch_size=16, do_eval_with_all_negative=False,
                                            pretrained_model="unixcoder", peft_alg="pret")


In [ ]:
!rm *.csv

### UniXCoder LoRA

In [ ]:
!mkdir unixcoder-lora

In [ ]:
codeflowlm.train.train_project_with_lat_ver(batch_classifier_dir, path, 'unixcoder-lora/.', commit_guru_path,
                                            full_features_train_file, full_features_valid_file,
                                            full_features_test_file, full_changes_train_file,
                                            full_changed_valid_file, full_changes_test_file,
                                            project='commons-scxml', early_stop_metric='gmean',
                                            adjust_th=False, skewed_oversample=True,
                                            batch_size=16, do_eval_with_all_negative=False,
                                            pretrained_model="unixcoder", peft_alg="lora")

In [ ]:
!rm *.csv

### CodeT5+

In [ ]:
!mkdir codet5p-lora

In [ ]:
codeflowlm.train.train_project_with_lat_ver(batch_classifier_dir, path, 'codet5p-lora/.', commit_guru_path,
                                            full_features_train_file, full_features_valid_file,
                                            full_features_test_file, full_changes_train_file,
                                            full_changed_valid_file, full_changes_test_file,
                                            project='commons-scxml', early_stop_metric='gmean',
                                            adjust_th=False, skewed_oversample=True,
                                            batch_size=16, do_eval_with_all_negative=False,
                                            pretrained_model="codet5p", peft_alg="lora")

In [ ]:
!rm *.csv

## Inference

In [ ]:
import pickle
import random

def get_features_for_prediction():
  full_changes_predict_file = full_changes_test_file
  project = 'commons-scxml'
  features_file = full_features_test_file

  with open(full_changes_predict_file, "rb") as f:
      full_changes_predict = pickle.load(f)

  with open(features_file, "rb") as f:
      features = pickle.load(f)

  idx = random.randint(0, len(full_changes_predict[0]) - 1)
  print("Randomly selected commit for first prediction:")
  print("Commit ID:", full_changes_predict[0][idx])
  print("Label:", full_changes_predict[1][idx])

  changes = [
      [full_changes_predict[0][idx]],
      [full_changes_predict[1][idx]],
      [full_changes_predict[2][idx]],
      [full_changes_predict[3][idx]],
  ]
  features = features.iloc[[idx]].reset_index(drop=True)
  return changes, features, project


In [ ]:
changes, features, project = get_features_for_prediction()

Randomly selected commit for first prediction:
Commit ID: 77868bf772a03d61701bc0d32dfafea63a0382c2
Label: 0.0


### UniXCoder PreT

In [ ]:
codeflowlm.predict.predict(batch_classifier_dir, path, changes, project, features, 'unixcoder-pret/.', th=0.5, pretrained_model="unixcoder", peft_alg="pret", batch_size=16, adjust_th=False)

### UniXCoder LoRA

In [ ]:
codeflowlm.predict.predict(batch_classifier_dir, path, changes, project, features, 'unixcoder-lora/.', th=0.5, pretrained_model="unixcoder", peft_alg="lora", batch_size=16, adjust_th=False)

### CodeT5+

In [ ]:
codeflowlm.predict.predict(batch_classifier_dir, path, changes, project, features, 'codet5p-lora/.', th=0.5, pretrained_model="codet5p", peft_alg="lora", batch_size=16, adjust_th=False)